## 1. Setup

In [1]:
import sys
sys.path.insert(0, '.')

from multi_persona_explainer import (
    load_model_artifacts, get_claim_context, generate_explanation, explain_claim
)
import pandas as pd
import numpy as np
from IPython.display import Markdown, display

artifacts = load_model_artifacts()
claims_df = artifacts['claims_df']
print(f'Loaded {len(claims_df)} claims')
print(f'Declined: {(claims_df["target"]==0).sum()}, Completed: {(claims_df["target"]==1).sum()}')

Loaded 2880 claims
Declined: 453, Completed: 2427


---
## 2. Demo — Declined Claim

Let's explain a claim that was **declined**. This is the most important case — customers want to know *why*.

In [2]:
# Find a declined claim with interesting characteristics
declined_indices = claims_df[claims_df['target'] == 0].index.tolist()
declined_idx = declined_indices[0]

ctx = get_claim_context(artifacts, declined_idx)
print(f'Claim #{declined_idx}')
print(f'  Prediction: {ctx["prediction"]} (P(approved) = {ctx["approval_probability"]:.3f})')
print(f'  Actual: {ctx["actual_status"]}')
print(f'  Type: {ctx["claim_type"]} | Coverage: {ctx["coverage"]}')
print(f'  Device: {ctx["make"]} {ctx["model_name"]} ({ctx["device_type"]})')
print(f'  Damage: {ctx["llm_damage_type"]} ({ctx["llm_damage_severity"]})')
print(f'  Incident: {ctx["llm_incident_type"]} | Gradual wear: {ctx["llm_is_gradual_wear"]}')
print(f'\n  Description: {ctx["issue_description"][:300]}...')
print(f'\n  Top SHAP factors:')
for f in ctx['top_factors'][:5]:
    print(f'    {f["feature"]}: {f["shap_value"]:+.4f} ({f["direction"]})')

Claim #13
  Prediction: Declined (P(approved) = 0.010)
  Actual: Declined
  Type: Accidental Damage | Coverage: ADLD
  Device: WUAWEI WUAWEI-AAA11 (SMARTPHONES)
  Damage: screen_crack (minor)
  Incident: drop | Gradual wear: False

  Description: *** blev pÃ¥sprungen av en person *** skulle till pendeltÃ¥get (huddinge station) nÃ¤r *** var pÃ¥ vÃ¤g fÃ¶r att kÃ¶pa lunch och tappade *** telefon. VarpÃ¥ en spricka uppstod pÃ¥ vÃ¤nster sida av skÃ¤rmen, fÃ¶rhÃ¥llandevis "mellan volymknapp och powerknapp" ...

  Top SHAP factors:
    policy_start_month: -1.8297 (toward decline)
    desc_length: -1.2836 (toward decline)
    device_issues_count: -1.0751 (toward decline)
    rrp: -1.0208 (toward decline)
    llm_has_police_report: -0.9166 (toward decline)


In [3]:
# Customer explanation
customer_exp = generate_explanation(ctx, 'customer')
display(Markdown(f'### Customer Explanation\n\n{customer_exp}'))

### Customer Explanation

# Claim Decision Letter

Dear Valued Customer,

Thank you for filing a claim for your WUAWEI WUAWEI-AAA11 smartphone. We understand how frustrating it is when your device gets damaged, and we appreciate you providing details about the incident at Huddinge Station.

**Unfortunately, we've had to decline your claim.**

After reviewing your submission, several factors contributed to this decision:

- The timing of your policy relative to when the damage occurred
- The information provided in your claim description
- Your device's claim history
- The device's original retail price
- The absence of a police report

While we can see this was an accidental drop that caused screen cracking—genuine accidents we normally cover—the combination of these factors means we're unable to approve it at this time.

**What you can do:**

You have the right to appeal this decision. If you believe we've made an error or if you can provide additional information (such as photographic evidence of the damage or purchase documentation), please contact us within 30 days. We're here to help.

We genuinely regret we couldn't assist with this claim.

Best regards,  
**Customer Care Team**

In [4]:
# Adjuster explanation
adjuster_exp = generate_explanation(ctx, 'adjuster')
display(Markdown(f'### Claims Adjuster Note\n\n{adjuster_exp}'))

### Claims Adjuster Note

**CLAIMS ADJUSTER NOTE**

**CLAIM DECISION SUMMARY**
- **Status:** DECLINED
- **ML Confidence:** 1% approval probability (threshold: 40%)
- **High confidence decline with minimal approval likelihood**

**SUPPORTING EVIDENCE FOR DECLINE**

• **Valid coverage trigger:** Accidental Damage (ADLD) applies to drop incidents
• **Clear incident documentation:** Detailed description of drop at Huddinge station while purchasing lunch
• **Minor damage severity:** Screen crack assessed as minor, located between volume/power buttons
• **Device remains functional:** Partial functionality retained post-damage

**KEY RISK FACTORS - MODEL CONCERNS**

• **Policy recency:** Started November (month 11) — strong negative weighting suggests early-stage claims flagged as higher risk
• **High device value:** RRP 17,790 SEK creates elevated claim exposure (excess 619 SEK = 3.5% of claim)
• **Missing documentation:** No police report filed — typical for minor drop incidents but flagged by model
• **Low description metrics:** Only 2 sentences may signal insufficient claim documentation

**INCONSISTENCIES FLAGGED**

⚠ **Technical concern:** Model heavily penalizes policy_start_month despite policy being active and in good standing. Early claims are common and legitimate.

⚠ **Severity mismatch:** "Minor" damage classification conflicts with screen functionality impact — crack location (between buttons) may affect usability more than severity rating suggests.

**MANUAL REVIEW RECOMMENDATION: YES**

This case warrants human review given:
1. Valid ADLD coverage for straightforward drop claim
2. Model over-reliance on policy recency rather than claim legitimacy
3. Clarification needed on actual device functionality post-crack
4. Customer provided coherent, detailed incident narrative

**Suggest:** Approve with excess or request damage photographs before final determination.

In [5]:
# Manager explanation
manager_exp = generate_explanation(ctx, 'manager')
display(Markdown(f'### Manager Summary\n\n{manager_exp}'))

### Manager Summary

# Claim Decision Summary

**Claim:** Accidental Damage – HUAWEI WUAWEI-AAA11 (Smartphones)

**Decision: DECLINE** | Confidence: 99%

This claim has been declined due to multiple risk indicators including recent policy inception, unusually high device value (£17,790), elevated device issue history, and absence of supporting documentation (no police report filed for claimed accidental damage). While the reported damage (minor screen crack from drop) appears straightforward, the combination of risk factors—particularly the lack of police report despite high claim value and policy recency—suggests potential claim validity concerns.

**Recommended Action: AUTO-DECLINE**

The exceptionally high confidence score (0.99) and consistent risk signals across multiple dimensions support automated decline without escalation. However, flag this claim for compliance review if the claimant disputes the decision, as the high device value warrants documented justification.

---
## 3. Demo — Approved Claim

A straightforward approved claim for comparison.

In [6]:
# Find an approved claim with high confidence
approved_indices = claims_df[claims_df['target'] == 1].index.tolist()
approved_idx = approved_indices[5]  # Pick one that's not the first

ctx_approved = get_claim_context(artifacts, approved_idx)
print(f'Claim #{approved_idx}')
print(f'  Prediction: {ctx_approved["prediction"]} (P(approved) = {ctx_approved["approval_probability"]:.3f})')
print(f'  Type: {ctx_approved["claim_type"]} | Damage: {ctx_approved["llm_damage_type"]}')
print(f'  Incident: {ctx_approved["llm_incident_type"]} | Gradual wear: {ctx_approved["llm_is_gradual_wear"]}')

Claim #5
  Prediction: Approved (P(approved) = 1.000)
  Type: Accidental Damage | Damage: screen_crack
  Incident: drop | Gradual wear: False


In [7]:
result_approved = explain_claim(artifacts, approved_idx)

for persona in ['customer', 'adjuster', 'manager']:
    display(Markdown(f'### {persona.title()} Explanation\n\n{result_approved[persona]}'))
    print()

### Customer Explanation

# Your Claim Has Been Approved ✓

Good news! We're pleased to let you know that your claim for accidental damage to your WUAWEI smartphone has been **approved**.

## What Led to Our Decision

Your claim met the requirements of your ADLD/THEFT coverage. Here's why we're moving forward:

- **Clear incident**: The drop damage is consistent with accidental damage covered under your policy
- **Device functionality**: Despite the cracked screen, your device is working properly (camera, audio, buttons, touchscreen all functioning)
- **Valid claim type**: Accidental damage from drops is covered under your plan
- **Reasonable cost**: The repair cost is proportionate to your device's value

## What Happens Next

We'll be in touch shortly with:
- Details about repair or replacement options
- Any applicable excess charges
- Timeline for resolution

We understand how frustrating it is when your device gets damaged, especially in an accident like this. We're here to help get you back up and running quickly.

If you have any questions about your claim, please don't hesitate to reach out.

### Adjuster Explanation

**INTERNAL CLAIMS ADJUSTER NOTE**

**Claim ID:** WUAWEI-AAA6 | **Decision:** APPROVED | **Confidence:** 100%

---

**PREDICTION SUMMARY**
ML model recommends approval with maximum confidence (1.0 vs. 0.40 threshold). Accidental damage claim for cracked screen following device drop qualifies under active ADLD/THEFT coverage.

---

**SUPPORTING EVIDENCE**
- **Damage Assessment:** Screen crack classified as minor severity; device remains partially functional across all critical systems (camera, audio, microphone, touchscreen, buttons operational)
- **Incident Clarity:** Drop incident is detailed and credible—device fell from vehicle; user was present
- **No Prior History:** Device previously undiagnosed for damage
- **Coverage Alignment:** Claim falls within ADLD policy scope; RRP 11,990 SEK with 1,149 SEK excess (9.6% ratio—reasonable threshold indicator)
- **User Accountability:** Self-use incident; no third-party liability complications

---

**RISK FLAGS & INCONSISTENCIES**
- **Model Bias Concern:** Primary approval driver is channel_encoded (SHAP +3.95), suggesting potential over-weighting of submission channel rather than claim substance
- **Description Quality:** Low uppercase ratio (5.6%) and presence of 2 questions in narrative flagged as decline factors by model, yet approval proceeded—indicates channel influence overrode content signals
- **Incomplete Redaction:** IMEI and location details partially obscured; verification of genuine device/incident location recommended before payment authorization

---

**MANUAL REVIEW RECOMMENDATION**
**LOW PRIORITY** – Approve and process. However, flag channel_encoded weighting for model recalibration review. Recommend standard fraud verification checks (IMEI validation, geographic consistency with SE policy) as routine due diligence.

**Payment Authorization:** Approved for 10,841 SEK (RRP minus excess).

### Manager Explanation

# Claim Decision Summary

**Claim Approved** – Minor screen crack from drop damage on a WUAWEI smartphone (device value: €11,990) with appropriate excess applied (€1,149). The high confidence score (1.0) is supported by legitimate approval factors including valid incident classification and reasonable excess-to-RRP ratio, though minor red flags exist in the description formatting (unusual capitalization and multiple questions) that weakly countered approval. No gradual wear detected and damage aligns with accidental damage coverage parameters.

**Recommended Action:** Auto-approve – Decision confidence is maximal with no material risk indicators (no gradual wear, high-value item has proportionate excess). Monitor description anomalies as potential pattern for future fraud detection tuning, but insufficient to warrant manual escalation on this case.

---
## 4. Demo — Borderline Case

A claim near the decision threshold — the most interesting case for adjuster review.

In [8]:
# Find a borderline claim (probability close to threshold)
config = artifacts['config']
threshold = config['optimal_threshold']
feature_cols = config['feature_columns']

X_all = claims_df[feature_cols].fillna(0).values
probas = artifacts['model'].predict_proba(X_all)[:, 1]

# Find claims closest to threshold
distances = np.abs(probas - threshold)
borderline_indices = np.argsort(distances)[:10]
borderline_idx = int(borderline_indices[0])

ctx_border = get_claim_context(artifacts, borderline_idx)
print(f'Claim #{borderline_idx} (borderline)')
print(f'  Prediction: {ctx_border["prediction"]} (P(approved) = {ctx_border["approval_probability"]:.3f}, threshold = {threshold})')
print(f'  Actual: {ctx_border["actual_status"]}')
print(f'  Type: {ctx_border["claim_type"]} | Damage: {ctx_border["llm_damage_type"]}')
print(f'  This claim is RIGHT at the decision boundary.')

Claim #2407 (borderline)
  Prediction: Approved (P(approved) = 0.682, threshold = 0.3999999999999998)
  Actual: Completed
  Type: Accidental Damage | Damage: screen_crack
  This claim is RIGHT at the decision boundary.


c:\Users\dainh\anaconda3\envs\bolt\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [9]:
result_border = explain_claim(artifacts, borderline_idx)

for persona in ['customer', 'adjuster', 'manager']:
    display(Markdown(f'### {persona.title()} Explanation (Borderline)\n\n{result_border[persona]}'))
    print()

### Customer Explanation (Borderline)

# Your Claim Decision

Good news! **Your claim has been approved.**

We understand how frustrating it must be to have your phone damaged in an accident like that. Based on the details you provided about the phone falling while you were sitting on the sofa, we can confirm this qualifies as accidental damage under your ADLD coverage.

**What happens next:**

Your claim for the HUAWEI HUAWEI-AAA104 smartphone will now move forward for processing. We'll be in touch shortly with:
- Details about repair or replacement options
- Information about any applicable excess
- Timeline for resolution

**About your screen damage:**

We've noted the moderate crack and line running across your display. This is consistent with the drop incident you described, and we're confident this falls within your coverage.

We appreciate you providing a clear description of what happened — it really does help us process claims quickly and fairly.

If you have any questions while we proceed, please don't hesitate to reach out. We're here to help.

Thank you for being a valued customer.

### Adjuster Explanation (Borderline)

**CLAIMS ADJUSTER NOTE**

**Claim Reference:** WUAWEI-AAA104 | **Decision:** APPROVED | **Confidence:** 68.2%

---

**PREDICTION SUMMARY**
ML model approved claim with moderate confidence (0.682 vs. 0.40 threshold). Accidental damage claim within ADLD coverage parameters, policy in grace period.

**SUPPORTING EVIDENCE**
- **Valid coverage:** ADLD explicitly covers accidental damage; device RRP €1,579
- **Clear incident:** Detailed description documents drop incident with specific damage outcome (screen crack + display malfunction)
- **Proportionate claim:** Excess (€59) represents 3.7% of RRP—reasonable ratio
- **Damage assessment:** Moderate-severity screen damage consistent with low-height drop narrative; partial device functionality appropriate to damage type
- **No gradual wear indicators:** Assessment confirms acute incident, not pre-existing degradation

---

**RISK FACTORS & INCONSISTENCIES**
- **Low description quality metrics:** Short description length (243 characters) and minimal capitalization (0.008 ratio) negatively weighted SHAP contributors, suggesting limited detail despite apparent clarity
- **Model frequency concern:** Low product code frequency (0.097) and model frequency (0.051) flagged by SHAP analysis—unusual device may indicate limited claims history for validation
- **Grace period status:** Policy recently activated; early claim timing warrants verification of policy start-to-incident timeline
- **Linguistic redaction:** Original Dutch text contains redacted content (asterisks); unverified whether redactions affect claim authenticity assessment

---

**RECOMMENDATION**
**Manual review advised** before final authorization. Verify:
1. Policy grace period terms and claim eligibility window
2. Redacted portions of claim description for completeness
3. Device ownership/purchase documentation
4. Repair quotation if proceeding

Approve subject to verification completion.

### Manager Explanation (Borderline)

# Claim Decision Summary

**Claim Approved** – Accidental damage claim for a €1,579 smartphone with moderate screen crack from drop damage. Model risk assessment supports approval (confidence: 68.2%), with no gradual wear detected and excess-to-RRP ratio within acceptable parameters. However, the borderline confidence score warrants attention; risk signals include atypical claim description formatting and product code frequency patterns that typically indicate higher decline risk. **Recommendation: Auto-approve with standard processing**, but flag for post-approval audit sampling given the moderate confidence threshold and conflicting risk indicators.

---
## 5. Prompt Engineering Strategy

### Why Three Personas?

Each stakeholder needs different information from the same prediction:

| Persona | Needs | Tone | Detail Level | Jargon |
|---------|-------|------|-------------|--------|
| **Customer** | Why was my claim approved/declined? What can I do? | Empathetic, clear | Low — plain language | None |
| **Claims Adjuster** | Is this prediction trustworthy? What evidence supports it? | Technical, analytical | High — specific data points | Insurance + ML terms OK |
| **Manager** | Should I review this? What's the risk? | Concise, executive | Summary only | Business terms |

### Grounding Strategy

Every explanation is grounded in **three layers of evidence**:

1. **SHAP values** — which features pushed the model toward approval or decline, and by how much
2. **LLM-extracted features** — structured damage assessment (type, severity, gradual wear, etc.)
3. **Raw claim data** — device info, policy details, financial data

This prevents hallucination — the LLM explains what the model actually decided, not what it imagines.

### Prompt Design Principles

1. **Role assignment**: Each prompt starts with a clear persona ("You are a friendly customer service representative" vs "You are writing an internal adjuster note")
2. **Context injection**: Claim data, SHAP factors, and LLM features are injected as structured data
3. **Output constraints**: Word limits, format requirements (bullets for adjuster, paragraphs for customer)
4. **Actionability**: Customer prompts require next-step guidance; adjuster prompts require risk flags
5. **Guardrails**: No SHAP numbers in customer output; no emotional language in adjuster output

---
## 6. Batch Generation — Sample Across Scenarios

In [10]:
# Generate explanations for a diverse set of claims
np.random.seed(42)

# Pick: 3 declined, 3 approved, 2 borderline
declined_sample = np.random.choice(declined_indices, 3, replace=False)
approved_sample = np.random.choice(approved_indices, 3, replace=False)
borderline_sample = borderline_indices[:2]

sample_indices = list(declined_sample) + list(approved_sample) + list(borderline_sample)

batch_results = []
for idx in sample_indices:
    idx = int(idx)
    result = explain_claim(artifacts, idx, personas=['customer', 'adjuster'])
    ctx = result['claim_context']
    batch_results.append({
        'claim_idx': idx,
        'prediction': ctx['prediction'],
        'actual': ctx['actual_status'],
        'probability': ctx['approval_probability'],
        'claim_type': ctx['claim_type'],
        'damage_type': ctx['llm_damage_type'],
        'customer_explanation': result['customer'],
        'adjuster_explanation': result['adjuster'],
    })
    print(f'  Claim #{idx}: {ctx["prediction"]} (P={ctx["approval_probability"]:.3f})')

batch_df = pd.DataFrame(batch_results)
print(f'\nGenerated {len(batch_df)} explanations')

  Claim #2232: Declined (P=0.004)
  Claim #356: Declined (P=0.006)
  Claim #2270: Declined (P=0.004)
  Claim #471: Approved (P=1.000)
  Claim #2644: Approved (P=1.000)
  Claim #1328: Approved (P=1.000)
  Claim #2407: Approved (P=0.682)
  Claim #2408: Approved (P=0.682)

Generated 8 explanations


In [11]:
# Display a few examples
for _, row in batch_df.head(4).iterrows():
    display(Markdown(f'---\n### Claim #{row["claim_idx"]} — {row["prediction"]} ({row["claim_type"]}, {row["damage_type"]})'))
    display(Markdown(f'**Customer:**\n\n{row["customer_explanation"]}'))
    display(Markdown(f'**Adjuster:**\n\n{row["adjuster_explanation"]}'))

---
### Claim #2232 — Declined (Accidental Damage, multiple_damage)

**Customer:**

# Claim Decision: Unfortunately Declined

Dear Valued Customer,

Thank you for filing your claim for your WUAWEI smartphone. I understand how frustrating it is when your device starts showing issues, and I appreciate you providing those details.

After careful review, **we've had to decline your claim**. Here's why:

The damage you're describing—the screen scratches, declining fingerprint scanner performance, and charging difficulties—appears to be **gradual wear and tear** rather than damage from a sudden accident. Our Accidental Damage coverage specifically protects against unexpected incidents (like dropping your phone), not damage that develops over time with normal use.

**What you can do:**

1. **Appeal this decision** - If you believe this damage resulted from a specific accidental incident rather than gradual wear, please provide additional details and we'll reconsider
2. **Contact us** - Our team can discuss other coverage options that might help with device repairs
3. **Check your warranty** - Your device manufacturer's warranty may cover some of these issues

I know this isn't the news you hoped for, and I'm sorry we couldn't help with this particular claim. If you'd like to discuss this further or explore your options, please don't hesitate to reach out—I'm here to help.

Best regards,
**Your Insurance Team**

**Adjuster:**

**CLAIMS ADJUSTER NOTE**

**Claim ID:** WUAWEI-AAA145 | **Decision:** DECLINED | **Confidence:** Very Low (0.4%)

---

**DECISION SUMMARY**
ML model recommends decline with extremely low approval probability (0.004 vs. 0.40 threshold). However, confidence level is critically weak, indicating high uncertainty in this classification.

---

**SUPPORTING EVIDENCE FOR DECLINE**
- **Gradual wear classification (SHAP: -4.66):** Primary driver. Claimant reports progressive deterioration (fingerprint scanner degradation, charging intermittency, screen scratches)—consistent with wear rather than acute damage
- **Device functionality:** Partial (not total loss), suggesting operational despite defects
- **Incident clarity:** Moderate—no specific damage event described

---

**CONTRADICTING FACTORS & RISK FLAGS**
- **Policy coverage mismatch:** ADLD/THEFT explicitly covers "Accidental Damage." Gradual wear typically excluded, but screen scratches may constitute covered accidental damage if distinct from wear
- **Linguistic signal weakness:** Low uppercase ratio and modest word count (28 words) negatively weighted—potentially penalizing legitimate claims with brief, matter-of-fact descriptions
- **Device age:** 304 days post-purchase; within typical wear manifestation period but doesn't automatically exclude coverage
- **User fault assessment:** Model confidence that user is NOT at fault (value=0) supports claim legitimacy
- **Claim value consideration:** RRP €1,579 with €59 excess—material claim warranting careful review

---

**MANUAL REVIEW RECOMMENDATION: YES**

The model's near-zero confidence (0.4%) combined with genuine coverage ambiguity (gradual wear vs. accidental damage distinction) warrants adjuster intervention. Screen scratches may be separately claimable despite wear components. Recommend:
1. Clarify if scratches resulted from specific incident
2. Assess whether fingerprint/charging issues predate policy
3. Verify ADLD policy wording on wear exclusions

---
### Claim #356 — Declined (Theft, theft)

**Customer:**

# Your Claim Decision

Dear Valued Customer,

Thank you for filing your claim for your HUAWEI smartphone that was stolen. I understand how frustrating this must be, and I appreciate you providing the details of what happened.

After careful review, we've made the difficult decision to **decline your claim**.

Here's why:

Our assessment found that several factors didn't align with our coverage requirements:
- The circumstances described in your claim (losing the phone after dinner, uncertainty about where it disappeared) raised questions about whether the loss meets our theft coverage criteria
- The claim documentation provided didn't contain enough detail for us to fully verify the theft occurred as covered
- The value versus repair cost ratio also factored into our decision

**What you can do:**

If you believe this decision is incorrect, you have the right to appeal. Please contact us with:
- Any additional evidence (police report, witness statements, CCTV footage from the venue)
- More details about exactly when and where the phone was last seen
- Any other information supporting that theft occurred

We're here to help. Feel free to reach out if you'd like to discuss this further or need clarification.

Best regards,
Customer Service Team

...

**Customer:**

# Your Claim Decision

Dear Valued Customer,

Thank you for filing a claim with us. We've carefully reviewed your case regarding the WUAWEI WUAWEI-AAA320, and unfortunately, we must **decline your claim at this time**.

## Why We Declined

Based on our assessment, we have some concerns:

1. **Unclear incident details** – The description of how the damage occurred isn't entirely clear to us, which makes it difficult to confirm this is covered under accidental damage.

2. **Software vs. hardware** – Our assessment suggests this may be a software issue rather than accidental physical damage, which falls outside your ADLD coverage.

## What You Can Do

We understand this is frustrating. Here are your options:

- **Provide more information** – If you can give us clearer details about what happened (including photos or videos), we'd be happy to re-review your claim
- **File an appeal** – You have the right to formally appeal this decision within 30 days
- **Contact us directly** – Our claims team can discuss whether other coverage options might apply

We're here to help. Please reach out if you'd like to discuss this further or provide additional details.

Best regards,
Your Insurance Team

**Adjuster:**

**CLAIMS ADJUSTER NOTE**

**Claim ID:** WUAWEI-AAA320 | **Decision:** DECLINED | **Confidence:** Very Low (0.4%)

---

**PREDICTION SUMMARY**
ML model recommends decline with extremely low approval probability (0.004 vs. 0.40 threshold). This marginal confidence level warrants caution in reliance on automated decision alone.

---

**SUPPORTING EVIDENCE FOR DECLINE**
- **Damage Classification:** Software issue (moderate severity) typically falls outside accidental damage scope; ADLD policies typically exclude software faults
- **Incident Clarity:** Description is vague and heavily redacted, with unclear causation ("unknown_cause" flagged)
- **User Attribution:** AI assessment indicates "User at fault: True," suggesting non-accidental origin
- **Linguistic Indicators:** Low uppercase ratio and vague phrasing suggest lower claim legitimacy confidence

---

**CONTRADICTING FACTORS & RISK ALERTS**
- **Critical Redaction Issues:** Significant text masking (*****) obscures actual incident details—prevents proper assessment of whether damage was truly accidental vs. device malfunction
- **Policy Validity:** Active policy with 366-day tenure and appropriate excess (€45 against €499 RRP) are in order
- **Ambiguous Causation:** "Telefoon lag op kast" (phone on cabinet) *could* indicate accidental fall, but description is too obscured to confirm
- **Model Confidence Crisis:** 0.4% approval probability is near-zero; this extreme reading suggests model uncertainty rather than clear decline evidence

---

**RECOMMENDATION**
**ESCALATE FOR MANUAL REVIEW.** The extreme low confidence combined with heavily redacted claim description makes automated decline inappropriate. Adjuster should:
1. Request unredacted claim description from claimant
2. Clarify incident mechanism (accidental drop vs. software malfunction)
3. Verify whether device damage is physical or software-only
4. Re-assess under proper incident clarity before final decision

---
### Claim #471 — Approved (Accidental Damage, screen_crack)

**Customer:**

# Your Claim Decision: **APPROVED** ✓

Dear Valued Customer,

Thank you for filing your claim with us. I'm pleased to let you know that your claim for accidental damage to your HUAWEI smartphone has been **approved**.

**What happened:**
We reviewed your claim for the cracked screen that occurred when your phone was in your pocket at work. This type of accidental damage is covered under your ADLD policy, and your claim meets all the necessary requirements for approval.

**What happens next:**
You should expect to hear from our claims team within the next 2-3 business days with details about:
- Repair or replacement options available to you
- Any applicable excess/deductible
- How to proceed with getting your device fixed

We understand how frustrating it is when your device gets damaged, especially during your workday. We're here to make this process as smooth as possible for you.

If you have any questions in the meantime, please don't hesitate to reach out. We're happy to help!

Best regards,  
**Customer Service Team**

**Adjuster:**

# CLAIMS ADJUSTER NOTE

**Claim ID:** WUAWEI-AAA123 | **Decision:** APPROVED | **Confidence:** 100%

---

## PREDICTION SUMMARY
ML model recommends approval with maximum confidence (1.0 probability vs. 0.40 threshold). Device valued at €18,407 with €589 excess. Accidental damage claim under active ADLD coverage in Sweden.

---

## SUPPORTING EVIDENCE
- **Policy Status:** Active, 181 days in force (low lapse risk)
- **Damage Assessment:** Screen crack, moderate severity, crush incident
- **User Accountability:** Acknowledged fault (occupational hazard—metal fabrication environment)
- **Device Functionality:** Partial—operational but compromised
- **Tone Analysis:** Matter-of-fact, credible presentation
- **Key Approval Drivers:** Policy duration (+2.11 SHAP), high RRP (+0.71 SHAP), incident classification (+0.93 SHAP)

---

## RISK FACTORS & INCONSISTENCIES
- **Critical:** Description contains character encoding corruption (Ã¤, *** masking) indicating potential OCR/transcription error. Readability severely compromised.
- **Vague Incident Clarity:** Claim narrative lacks specific temporal/spatial detail. Mechanism unclear ("*** fick tryck på"—incomplete text).
- **Occupational Context:** Metal fabrication environment increases crush risk plausibility but warrants verification of workplace conditions.
- **Conflicting Signals:** Moderate damage severity penalizes decision (-0.38 SHAP) yet approval holds; low word count (-0.69 SHAP) suggests incomplete description.

---

## RECOMMENDATION
**MANUAL REVIEW REQUIRED** before final settlement. Despite strong model confidence, the corrupted claim description undermines proper verification. Recommend:
1. Request legible written statement from claimant
2. Obtain photographic evidence of damage
3. Verify occupational crush mechanism plausibility
4. Confirm policy coverage terms for workplace-related incidents

**Proceed cautiously.** High RRP increases loss exposure; documentation gaps must be resolved.

---
## 7. Save Outputs

In [12]:
from pathlib import Path
output_dir = Path('../../data')
batch_df.to_csv(output_dir / 'sample_explanations.csv', index=False)
print(f'Saved sample explanations to {output_dir / "sample_explanations.csv"}')
print(f'Columns: {list(batch_df.columns)}')

Saved sample explanations to ..\..\data\sample_explanations.csv
Columns: ['claim_idx', 'prediction', 'actual', 'probability', 'claim_type', 'damage_type', 'customer_explanation', 'adjuster_explanation']


---
## Wrap up

The explanation pipeline works. For each claim it pulls the SHAP values and LLM-extracted features, builds a persona-specific prompt, and sends it to Claude. Three personas:
- Customer gets a friendly explanation with next steps
- Adjuster gets SHAP numbers, evidence bullets, and risk flags
- Manager gets a 3-sentence summary with escalation recommendation

The key thing is grounding - every explanation references actual model outputs, not made-up reasoning. If SHAP says incident_type pushed toward decline, the explanation mentions that.

The `multi_persona_explainer.py` module is designed to plug straight into the FastAPI endpoint.